# TT/MPS基礎 07 — Orthogonality Center と QR による Center 移動

## 今回の位置づけ

前回は、3階 TT/MPS を

$$
X=G_1^{[L]}G_2^{[C]}G_3^{[R]}
$$

という mixed-canonical form に変換し、

$$
\|X\|_F=\|G_2^{[C]}\|_F
$$

および

$$
\|\Delta X\|_F=\|\Delta G_2^{[C]}\|_F
$$

を確認しました。

今回は、その $G_2^{[C]}$ を **orthogonality center** として理解し、さらに QR によって center を

$$
2\rightarrow3
$$

および

$$
2\rightarrow1
$$

へ移動します。

### 今回やること

1. orthogonality center の意味を整理する
2. ボンド添字 $\alpha_1,\alpha_2$ が左右の正規直交基底ラベルになる意味を確認する
3. center コアが「左基底 × 物理基底 × 右基底」の係数テンソルであることを理解する
4. center を $2\rightarrow3$ に QR で移動する
5. center を $2\rightarrow1$ に QR で移動する
6. 移動後も全テンソル不変性・直交性・ノルム局所化が保たれることを数値確認する
7. 最後に一般の $d$ コア TT/MPS における center の位置づけを整理する

### 今回はまだ扱わないもの

- SVD を使う center 移動
- Schmidt 分解
- 特異値
- entanglement
- bond center
- TT rounding
- TT-matrix / MPO
- DMRG / ALS / 局所最適化
- QR の rank 条件や rank-deficient な場合

今回も、理論と検証条件までは示しますが、**実装部分は `TODO` として残します。**

## 1. Orthogonality Center とは何か

第2サイトを中心とした mixed-canonical form は

$$
X=G_1^{[L]}G_2^{[C]}G_3^{[R]}
$$

です。

shape は

$$
G_1^{[L]}\in\mathbb{R}^{1\times n_1\times r_1},\qquad
G_2^{[C]}\in\mathbb{R}^{r_1\times n_2\times r_2},\qquad
G_3^{[R]}\in\mathbb{R}^{r_2\times n_3\times1}.
$$

この配置では、

- center より左の $G_1^{[L]}$ は左直交
- center より右の $G_3^{[R]}$ は右直交
- center の $G_2^{[C]}$ 自身には、一般には左・右どちらの直交条件も課さない

という役割分担になります。

$$
\underbrace{G_1^{[L]}}_{\text{左の直交化部分}}
\quad
\underbrace{G_2^{[C]}}_{\text{orthogonality center}}
\quad
\underbrace{G_3^{[R]}}_{\text{右の直交化部分}}
$$

重要なのは、**center 自身を直交化したいわけではない**という点です。
左右を正規直交基底として整え、その間に残る一般の係数テンソルが orthogonality center です。

## 2. ボンド添字は左右の基底ラベル

左部分収縮を

$$
L_1(i_1,\alpha_1)=G_1^{[L]}(1,i_1,\alpha_1)
$$

とすると、

$$
L_1^TL_1=I_{r_1}.
$$

したがって、

$$
|L_{\alpha_1}\rangle=
\sum_{i_1=1}^{n_1}L_1(i_1,\alpha_1)|i_1\rangle
$$

は正規直交基底です。

つまり、

$$
\alpha_1\Longleftrightarrow\text{左ブロックの正規直交基底ラベル}
$$

です。

同様に、

$$
R_2(\alpha_2,i_3)=G_3^{[R]}(\alpha_2,i_3,1)
$$

は

$$
R_2R_2^T=I_{r_2}
$$

を満たすので、

$$
|R_{\alpha_2}\rangle=
\sum_{i_3=1}^{n_3}R_2(\alpha_2,i_3)|i_3\rangle
$$

も正規直交基底です。

よって、

$$
\alpha_2\Longleftrightarrow\text{右ブロックの正規直交基底ラベル}
$$

です。

したがって中心コアの成分

$$
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
$$

は、

$$
\boxed{\text{左基底}\times\text{第2サイトの物理基底}\times\text{右基底}}
$$

に対する係数です。

## 3. 中心コアは全テンソルの局所座標

左右部分収縮を使うと、

$$
X(i_1,i_2,i_3)=
\sum_{\alpha_1,\alpha_2}
L_1(i_1,\alpha_1)
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
R_2(\alpha_2,i_3)
$$

です。

状態ベクトル風に書けば、

$$
|X\rangle=
\sum_{\alpha_1,i_2,\alpha_2}
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
|L_{\alpha_1}\rangle\otimes|i_2\rangle\otimes|R_{\alpha_2}\rangle.
$$

つまり $G_2^{[C]}$ は、

$$
\mathbb{R}^{r_1}\otimes\mathbb{R}^{n_2}\otimes\mathbb{R}^{r_2}
$$

という座標系における $X$ の係数テンソルです。

左右基底が正規直交なので、

$$
\|X\|_F=\|G_2^{[C]}\|_F
$$

が成り立ちます。

これが、center にノルムや局所的な変化量が集約される理由です。

### ノルム局所化を途中式で確認する

mixed-canonical form

$$
X(i_1,i_2,i_3)
=
\sum_{\alpha_1,\alpha_2}
L_1(i_1,\alpha_1)
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
R_2(\alpha_2,i_3)
$$

から出発します。

Frobenius ノルムの二乗は、

$$
\|X\|_F^2
=
\sum_{i_1,i_2,i_3}
\left[
\sum_{\alpha_1,\alpha_2}
L_1(i_1,\alpha_1)
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
R_2(\alpha_2,i_3)
\right]^2.
$$

2つ目のコピーのボンド添字を
$\beta_1,\beta_2$
とすると、

$$
\begin{aligned}
\|X\|_F^2
&=
\sum_{\substack{
\alpha_1,\beta_1\\
\alpha_2,\beta_2\\
i_1,i_2,i_3
}}
L_1(i_1,\alpha_1)
L_1(i_1,\beta_1)
\\
&\qquad\cdot
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
G_2^{[C]}(\beta_1,i_2,\beta_2)
\\
&\qquad\cdot
R_2(\alpha_2,i_3)
R_2(\beta_2,i_3).
\end{aligned}
$$

ここで左直交性

$$
\sum_{i_1}
L_1(i_1,\alpha_1)
L_1(i_1,\beta_1)
=
\delta_{\alpha_1\beta_1}
$$

と、右直交性

$$
\sum_{i_3}
R_2(\alpha_2,i_3)
R_2(\beta_2,i_3)
=
\delta_{\alpha_2\beta_2}
$$

を使うと、異なるボンド基底どうしのクロス項が消えます。

したがって、

$$
\begin{aligned}
\|X\|_F^2
&=
\sum_{\alpha_1,i_2,\alpha_2}
\left(
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
\right)^2
\\
&=
\|G_2^{[C]}\|_F^2.
\end{aligned}
$$

よって、

$$
\boxed{
\|X\|_F
=
\|G_2^{[C]}\|_F
}
$$

です。

このため、center を移した後も

$$
\boxed{
\text{center の左：左直交}
\qquad
\text{center の右：右直交}
}
$$

という mixed-canonical 条件を保つことが重要です。

## 4. 今回の数値検証用セットアップ

ここからは、前回までに学習済みの処理を使って、

$$
X=G_1^{[L]}G_2^{[C]}G_3^{[R]}
$$

を作ります。

このセットアップ自体は今回の課題ではありません。
以後は、この `G1_left`, `G2_center`, `G3_right` を出発点として center を左右へ移動します。

In [1]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(G1: torch.Tensor, G2: torch.Tensor, G3: torch.Tensor) -> torch.Tensor:
    """3個のTTコアから3階テンソルを再構成する。"""
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


# 小さい3階TT
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3

G1 = torch.randn(1, n1, r1)
G2 = torch.randn(r1, n2, r2)
G3 = torch.randn(r2, n3, 1)

X_before = reconstruct_tt3(G1, G2, G3)

# --- 06 までに学習済みの mixed-canonical 化 ---
A1 = G1.squeeze(0)
Q1, R1 = torch.linalg.qr(A1, mode="reduced")
G1_left = Q1.unsqueeze(0)

A3 = G3.squeeze(-1)
Q3, R3 = torch.linalg.qr(A3.T, mode="reduced")
G3_right = Q3.T.unsqueeze(-1)

G2_tmp = torch.tensordot(R1, G2, dims=([1], [0]))
G2_center = torch.tensordot(G2_tmp, R3.T, dims=([2], [0]))

X_center2 = reconstruct_tt3(G1_left, G2_center, G3_right)

print("G1_left  :", tuple(G1_left.shape))
print("G2_center:", tuple(G2_center.shape))
print("G3_right :", tuple(G3_right.shape))
print("center-2 reconstruction error =", torch.linalg.norm(X_center2 - X_before).item())


G1_left  : (1, 4, 2)
G2_center: (2, 3, 3)
G3_right : (3, 5, 1)
center-2 reconstruction error = 8.426803138067863e-15


## 5. Center を右へ移す：$2\rightarrow3$

現在は

$$
X=G_1^{[L]}G_2^{[C]}G_3^{[R]}.
$$

center を第3サイトへ移すと、目標は

$$
\boxed{X=G_1^{[L]}G_2^{[L]}G_3^{[C]}}.
$$

旧 center の $G_2^{[C]}$ は、新しい center の**左側**に残ります。
したがって、$G_2^{[C]}$ を左直交化します。

### 左展開

$$
G_2^{[C]\langle L\rangle}
\in\mathbb{R}^{(r_1n_2)\times r_2}.
$$

これに reduced QR を適用して、

$$
G_2^{[C]\langle L\rangle}=QR
$$

とします。

$Q$ を reshape して

$$
G_2^{[L]}\in\mathbb{R}^{r_1\times n_2\times r_2}
$$

を作ります。

残った $R$ は右隣へ吸収し、

$$
G_3^{[C]}=R\,G_3^{[R]}
$$

とします。

これで center は第3サイトへ移ります。

### なぜ $R$ を右隣へ吸収しても $X$ は変わらないか

QR 分解より、

$$
G_2^{[C]\langle L\rangle}
=
QR
$$

なので、テンソルへ戻せば

$$
G_2^{[C]}
=
G_2^{[L]}R
$$

と考えられます。

したがって、

$$
\begin{aligned}
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
&=
G_1^{[L]}
\left(
G_2^{[L]}R
\right)
G_3^{[R]}
\\
&=
G_1^{[L]}
G_2^{[L]}
\left(
RG_3^{[R]}
\right).
\end{aligned}
$$

そこで、

$$
\boxed{
G_3^{[C]}
=
RG_3^{[R]}
}
$$

と定義すれば、全テンソルは変わりません。

これは近似ではなく、QR で分けた因子を隣へ移しているだけの gauge 変換です。

### reshape の対応

左展開では、

$$
(\alpha_1,i_2,\alpha_2)
\longmapsto
((\alpha_1,i_2),\alpha_2)
$$

とまとめるので、行列 shape は

$$
(r_1n_2)\times r_2
$$

です。

PyTorch では `G2_center.reshape(r1 * n2, r2)` に対応します。


## 6. 演習1 — Center を $2\rightarrow3$ へ移す

### TODO

1. `G2_center` を左展開する
2. reduced QR を適用する
3. $Q$ を reshape して `G2_left` を作る
4. $R$ を `G3_right` の左ボンドへ吸収して `G3_center` を作る
5. shape を確認する
6. 新しい TT から全テンソルを再構成する
7. 移動前後の再構成誤差を確認する

期待する形は、

$$
G_1^{[L]}\quad G_2^{[L]}\quad G_3^{[C]}
$$

です。

### 実装前に考えること

- `G2_center` のどの軸をまとめれば左展開になるか
- QR の $R$ は、なぜ `G3_right` の左ボンドへ吸収するのか
- center が第3サイトへ移った後、$G_3^{[C]}$ に直交条件を課す必要があるか

In [2]:
# TODO 1:
# center を 2 -> 3 に移してください。
#
# 1. G2_center を左展開
C_left = G2_center.reshape(r1*n2,r2)
#
# 2. reduced QR
Q2_move_right, R2_move_right = torch.linalg.qr(C_left, mode="reduced")
#
# 3. Q を G2_left に戻す
G2_left = Q2_move_right.reshape(r1, n2, r2)
#
# 4. R を G3_right の左ボンドへ吸収
print(R2_move_right.shape,G3_right.shape)

G3_center = torch.tensordot(R2_move_right,G3_right,dims=([1], [0]))
#
# 5. shape を確認
print("G2_left shape  :",G2_left.shape)
print("G3_center shape:",G3_center.shape)
#
# 6. 全テンソルを再構成
X_center3 = reconstruct_tt3(G1_left,G2_left,G3_center)
#
# 7. 移動前後の差
move_right_reconstruction_error = torch.norm(X_center3-X_center2)
print("2 -> 3 reconstruction error =",move_right_reconstruction_error)

print("TODO: center を 2 -> 3 に移す")


torch.Size([3, 3]) torch.Size([3, 5, 1])
G2_left shape  : torch.Size([2, 3, 3])
G3_center shape: torch.Size([3, 5, 1])
2 -> 3 reconstruction error = tensor(9.5680e-15)
TODO: center を 2 -> 3 に移す


## 7. 演習2 — $2\rightarrow3$ 移動後の性質を確認する

center を第3サイトへ移した後は、

$$
G_1^{[L]}\quad G_2^{[L]}\quad G_3^{[C]}
$$

です。

確認したいのは次の3点です。

### 1. 第2コアの左直交性

$$
\left(G_2^{[L]\langle L\rangle}\right)^T
G_2^{[L]\langle L\rangle}=I_{r_2}.
$$

### 2. 全テンソル不変性

$$
\|X_{\mathrm{center2}}-X_{\mathrm{center3}}\|_F
$$

が丸め誤差水準になることを確認します。

### 3. ノルム局所化

center が第3サイトなので、

$$
\boxed{\|X\|_F=\|G_3^{[C]}\|_F}
$$

になるはずです。

In [9]:
# TODO 2:
# center を 2 -> 3 に移した後の性質を確認してください。
#
# 1. G2_left の左展開
G2_left_matrix = G2_left.reshape(r1 * n2, r2)
#
# 2. 左直交性
left_gram = G2_left_matrix.T @ G2_left_matrix
I_r2 = torch.eye(left_gram.shape[1], dtype=left_gram.dtype, device=left_gram.device)
left_orthogonality_error = torch.linalg.vector_norm(left_gram - I_r2).item()
#
# 3. 全テンソル不変性
reconstruction_error_center3 = torch.linalg.vector_norm(X_center3 - X_center2).item()
#
# 4. ノルム局所化
X_norm = torch.linalg.vector_norm(X_center3).item()
G3_center_norm = torch.linalg.vector_norm(G3_center).item()
center3_norm_error = abs(X_norm - G3_center_norm)

print("G2 left orthogonality error =", left_orthogonality_error)
print("center-3 reconstruction error =", reconstruction_error_center3)
print("||X||_F - ||G3_center||_F =", center3_norm_error)


G2 left orthogonality error = 3.2016811410571575e-16
center-3 reconstruction error = 9.56802769522031e-15
||X||_F - ||G3_center||_F = 3.552713678800501e-15


## 8. Center を左へ移す：$2\rightarrow1$

次は再び、元の第2サイト中心の形

$$
X=G_1^{[L]}G_2^{[C]}G_3^{[R]}
$$

から始めます。

center を第1サイトへ移すと、目標は

$$
\boxed{X=G_1^{[C]}G_2^{[R]}G_3^{[R]}}.
$$

今回は旧 center の $G_2^{[C]}$ が、新しい center の**右側**に残ります。
したがって、$G_2^{[C]}$ を右直交化します。

### 右展開

$$
C=G_2^{[C]\langle R\rangle}
\in\mathbb{R}^{r_1\times(n_2r_2)}.
$$

右直交では、行を正規直交化したいので、

$$
C^T
$$

へ reduced QR を適用します。

$$
C^T=QR.
$$

すると、

$$
C=R^TQ^T.
$$

$Q^T$ を reshape して $G_2^{[R]}$ を作り、残った $R^T$ を左隣へ吸収して

$$
G_1^{[C]}=G_1^{[L]}R^T
$$

とします。

これで center は第1サイトへ移ります。

### なぜ $R^T$ を左隣へ吸収しても $X$ は変わらないか

右展開行列を

$$
C
=
G_2^{[C]\langle R\rangle}
$$

とし、

$$
C^T=QR
$$

とします。

転置を戻すと、

$$
C
=
R^TQ^T.
$$

$Q^T$ を右直交コアとして reshape すれば、

$$
G_2^{[C]}
=
R^TG_2^{[R]}
$$

と考えられます。

したがって、

$$
\begin{aligned}
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
&=
G_1^{[L]}
\left(
R^TG_2^{[R]}
\right)
G_3^{[R]}
\\
&=
\left(
G_1^{[L]}R^T
\right)
G_2^{[R]}
G_3^{[R]}.
\end{aligned}
$$

そこで、

$$
\boxed{
G_1^{[C]}
=
G_1^{[L]}R^T
}
$$

と定義すれば、全テンソルは変わりません。

### reshape の対応

右展開では、

$$
(\alpha_1,i_2,\alpha_2)
\longmapsto
(\alpha_1,(i_2,\alpha_2))
$$

とまとめるので、行列 shape は

$$
r_1\times(n_2r_2)
$$

です。

PyTorch の連続メモリ順では `G2_center.reshape(r1, n2 * r2)` がこの右展開に対応します。

ただし、**TODO の QR 実装自体は自分で書いてください。**


## 9. 演習3 — Center を $2\rightarrow1$ へ移す

### TODO

1. `G2_center` を右展開する
2. **転置した行列**へ reduced QR を適用する
3. $Q^T$ を reshape して `G2_right` を作る
4. $R^T$ を `G1_left` の右ボンドへ吸収して `G1_center` を作る
5. shape を確認する
6. 全テンソルを再構成する
7. 移動前後の差を確認する

### 実装前に考えること

通常の QR は $Q$ の**列**を直交化します。

しかし右直交で欲しいのは、

$$
G_2^{[R]\langle R\rangle}
\left(G_2^{[R]\langle R\rangle}\right)^T=I
$$

という**行直交**です。

したがって、なぜ

$$
C^T=QR
$$

とする必要があるのかを確認してから実装してください。

In [6]:
# TODO 3:
# center を 2 -> 1 に移してください。
#
# 1. G2_center を右展開
C_right = G2_center.reshape(r1,n2*r2)
#
# 2. C_right.T に reduced QR
Q2_move_left, R2_move_left = torch.linalg.qr(C_right.T, mode="reduced")
#
# 3. Q.T を G2_right に戻す
G2_right = Q2_move_left.T.reshape(r1, n2, r2)
#
# 4. R.T を G1_left の右ボンドへ吸収
G1_center = G1_center = torch.tensordot(G1_left, R2_move_left.T, dims=([2], [0]))
#
# 5. shape を確認
print("G1_center shape:", G1_center.shape)
print("G2_right shape :", G2_right.shape)
#
# 6. 全テンソルを再構成

X_center1 = reconstruct_tt3(G1_center,G2_right,G3_right)
#
# 7. 移動前後の差
move_left_reconstruction_error = torch.norm(X_center1-X_center2)
print("2 -> 1 reconstruction error =", move_left_reconstruction_error)

print("TODO: center を 2 -> 1 に移す")


G1_center shape: torch.Size([1, 4, 2])
G2_right shape : torch.Size([2, 3, 3])
2 -> 1 reconstruction error = tensor(7.1253e-15)
TODO: center を 2 -> 1 に移す


## 10. 演習4 — $2\rightarrow1$ 移動後の性質を確認する

center を第1サイトへ移した後は、

$$
G_1^{[C]}\quad G_2^{[R]}\quad G_3^{[R]}
$$

です。

確認するのは次の3点です。

### 1. 第2コアの右直交性

$$
G_2^{[R]\langle R\rangle}
\left(G_2^{[R]\langle R\rangle}\right)^T=I_{r_1}.
$$

### 2. 全テンソル不変性

$$
\|X_{\mathrm{center2}}-X_{\mathrm{center1}}\|_F
$$

が丸め誤差水準になることを確認します。

### 3. ノルム局所化

center が第1サイトなので、

$$
\boxed{\|X\|_F=\|G_1^{[C]}\|_F}
$$

になるはずです。

In [7]:
# TODO 4:
# center を 2 -> 1 に移した後の性質を確認してください。
#
# 1. G2_right の右展開
G2_right_matrix = G2_right.reshape(r1, n2 * r2)
#
# 2. 右直交性
right_gram = G2_right_matrix @ G2_right_matrix.T
I_r1 = torch.eye(right_gram.shape[0], dtype=right_gram.dtype, device=right_gram.device)
right_orthogonality_error = torch.linalg.vector_norm(right_gram - I_r1).item()
#
# 3. 全テンソル不変性
reconstruction_error_center1 = torch.linalg.vector_norm(X_center1 - X_center2).item()
#
# 4. ノルム局所化
X_norm = torch.linalg.vector_norm(X_center1).item()
G1_center_norm = torch.linalg.vector_norm(G1_center).item()
center1_norm_error = abs(X_norm - G1_center_norm)

print("G2 right orthogonality error =", right_orthogonality_error)
print("center-1 reconstruction error =", reconstruction_error_center1)
print("||X||_F - ||G1_center||_F =", center1_norm_error)


G2 right orthogonality error = 4.84396924379029e-16
center-1 reconstruction error = 7.125294692149989e-15
||X||_F - ||G1_center||_F = 3.552713678800501e-15


## 11. 一般の $d$ コア TT/MPS

一般の TT/MPS を

$$
X(i_1,\ldots,i_d)=
\sum_{\alpha_1,\ldots,\alpha_{d-1}}
G_1(1,i_1,\alpha_1)\cdots G_d(\alpha_{d-1},i_d,1)
$$

とします。

各コアの shape は

$$
G_k\in\mathbb{R}^{r_{k-1}\times n_k\times r_k},
\qquad r_0=r_d=1.
$$

サイト $c$ を orthogonality center に置く mixed-canonical form は、

$$
\boxed{
X=G_1^{[L]}\cdots G_{c-1}^{[L]}G_c^{[C]}G_{c+1}^{[R]}\cdots G_d^{[R]}
}
$$

です。

条件は、次の3つです。

$$
\boxed{
k<c
\quad\Longrightarrow\quad
G_k^{[L]}
\text{ は左直交}
}
$$

$$
\boxed{
k=c
\quad\Longrightarrow\quad
G_c^{[C]}
\text{ は一般の中心コア}
}
$$

$$
\boxed{
k>c
\quad\Longrightarrow\quad
G_k^{[R]}
\text{ は右直交}
}
$$

つまり、

$$
\boxed{
\begin{aligned}
k<c
&\Longrightarrow
G_k^{[L]}\text{ は左直交},\\
k=c
&\Longrightarrow
G_c^{[C]}\text{ は center コアであり、一般に直交条件を課さない},\\
k>c
&\Longrightarrow
G_k^{[R]}\text{ は右直交}.
\end{aligned}
}
$$

です。

左側をすべて縮約したブロック $L_{c-1}$ と、右側をすべて縮約したブロック $R_c$ は、それぞれ

$$
L_{c-1}^TL_{c-1}=I,
\qquad
R_cR_c^T=I
$$

という等長性を持ちます。

そのため、

$$
\boxed{\|X\|_F=\|G_c^{[C]}\|_F}
$$

です。

つまり、center がどのサイトにあっても、

$$
\boxed{
\text{center 左：左直交}
\qquad
\text{center：一般の係数テンソル}
\qquad
\text{center 右：右直交}
}
$$

という構造は同じです。

## 12. Center を一つ動かす一般則

center がサイト $c$ にあるとします。

### 右へ動かす

中心コアを左展開して

$$
G_c^{[C]\langle L\rangle}=QR
$$

とします。

すると、

$$
G_c^{[L]\langle L\rangle}=Q,
\qquad
G_{c+1}^{[C]}=R\,G_{c+1}^{[R]}.
$$

したがって、

$$
\cdots G_c^{[C]}G_{c+1}^{[R]}\cdots
$$

から

$$
\cdots G_c^{[L]}G_{c+1}^{[C]}\cdots
$$

へ center が一つ右へ移ります。

---

### 左へ動かす

中心コアを右展開して

$$
C=G_c^{[C]\langle R\rangle}
$$

とします。

行を直交化するため、

$$
C^T=QR.
$$

したがって、

$$
G_c^{[R]\langle R\rangle}=Q^T,
\qquad
G_{c-1}^{[C]}=G_{c-1}^{[L]}R^T.
$$

これによって、

$$
\cdots G_{c-1}^{[L]}G_c^{[C]}\cdots
$$

から

$$
\cdots G_{c-1}^{[C]}G_c^{[R]}\cdots
$$

へ center が一つ左へ移ります。

## 13. 今回の到達点

このNotebookで確認する流れは、

$$
\boxed{
\begin{aligned}
\text{mixed-canonical form}
&\Longrightarrow\text{orthogonality center}\\
&\Longrightarrow\text{左右ブロックが正規直交基底}\\
&\Longrightarrow\text{center コアが局所係数テンソル}\\
&\Longrightarrow\|X\|_F=\|G_c^{[C]}\|_F\\
&\Longrightarrow\text{QR で center を左右へ移動}
\end{aligned}
}
$$

数値的には、

### $2\rightarrow3$

- 全テンソル不変
- $G_2^{[L]}$ が左直交
- $\|X\|_F=\|G_3^{[C]}\|_F$

### $2\rightarrow1$

- 全テンソル不変
- $G_2^{[R]}$ が右直交
- $\|X\|_F=\|G_1^{[C]}\|_F$

を自分で確認できれば完了です。

ここでは **QR による site-center の移動まで**で止めます。

SVD、Schmidt 分解、特異値、entanglement、TT rounding、MPO、DMRG にはまだ進みません。